# Gold de continuidade (DEC/FEC) — V-Energy

Calcula DEC, FEC e CHI por alimentador e competência a partir das ocorrências de rede e do cadastro de UCs. A transformação inteira está neste notebook (sem `%run` e sem módulo `.py` externo). Produz as três tabelas do contrato (`contracts/tabelas/`):

1. `dim_alimentador`: um alimentador por linha, com o município e o número de UCs do cadastro.
2. `ocorrencias_uc`: uma ocorrência (UC interrompida) por linha, com a duração efetiva e a marcação de sobreposição calculadas na Silver.
3. `continuidade_alimentador_mensal`: alimentador × competência, com CHI, interrupções, número de UCs, DEC, FEC, causa principal e marcação de evento atípico. Todo alimentador aparece em toda competência do período; um mês sem ocorrência entra com zero.

**DEC** = CHI ÷ número de UCs do alimentador; **FEC** = interrupções ÷ número de UCs. Para um período maior, some CHI e interrupções das competências e divida pelo número de UCs, nunca tire a média dos DECs mensais. Excluir o evento atípico é filtrar as linhas com `flag_evento_atipico`.

Regras e dicionário de dados: [`docs/gold/continuidade.md`](../../docs/gold/continuidade.md).

O padrão `modo=validar` não grava nada: **Run all** valida as fontes, monta e confere as três tabelas em memória. Para gravar em `v-energy.gold`, o Job passa `modo=publicar` (ou troque o widget antes de executar), como na Gold de faturas.

In [0]:
"""Transformações e validações da Gold de continuidade (DEC/FEC)."""

from decimal import Decimal

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, IntegerType

PADRAO_ID_UC = r'^UC\d{6}\z'
PADRAO_ALIMENTADOR = r'^ALM-[A-Z]{3}-\d{3}\z'
# Causa de uma competência em que o alimentador não teve nenhuma interrupção
CAUSA_SEM_OCORRENCIA = 'SEM OCORRENCIA'
DECIMAL_CONTRATO = DecimalType(18, 2)
COLUNAS_OCORRENCIAS = [
    'id_ocorrencia',
    'id_uc',
    'alimentador',
    'causa',
    'data_abertura',
    'data_encerramento',
    'competencia',
    'duracao_min',
    'duracao_efetiva_min',
    'flag_sobreposicao',
    'flag_evento_atipico',
]
COLUNAS_CADASTRO = ['id_uc', 'alimentador', 'municipio', 'data_ligacao']


def contar(df: DataFrame, regras: dict) -> dict:
    """Conta, numa única passada, as linhas que violam cada regra."""
    linha = df.agg(*[F.count(F.when(cond, 1)).alias(nome) for nome, cond in regras.items()])
    return linha.first().asDict()


def checar(violacoes: dict, rotulo: str) -> None:
    """Imprime cada regra e falha se alguma tiver violação."""
    for regra, n in violacoes.items():
        print(f'[{"FALHA" if n else "OK   "}] ({rotulo}) {regra}: {n:,}')
    falhas = {regra: n for regra, n in violacoes.items() if n}
    if falhas:
        raise RuntimeError(f'Validações bloqueantes falharam ({rotulo}): {falhas}')


def fora_do_padrao(coluna: str, padrao: str):
    return F.col(coluna).isNull() | ~F.col(coluna).rlike(padrao)


def validar_fontes(ocorrencias: DataFrame, cadastro: DataFrame) -> None:
    """Colunas, chaves, domínios e cruzamento das duas Silvers, antes de qualquer escrita."""
    for df, colunas, nome in (
        (ocorrencias, COLUNAS_OCORRENCIAS, 'ocorrências'),
        (cadastro, COLUNAS_CADASTRO, 'cadastro'),
    ):
        ausentes = sorted(set(colunas) - set(df.columns))
        if ausentes:
            raise RuntimeError(f'Silver de {nome} sem as colunas {ausentes}')

    violacoes = contar(
        ocorrencias,
        {
            'ocorrências: id_ocorrencia nulo': F.col('id_ocorrencia').isNull(),
            'ocorrências: id_uc fora do padrão UC + 6 dígitos': fora_do_padrao(
                'id_uc', PADRAO_ID_UC
            ),
            'ocorrências: alimentador fora do padrão ALM-AAA-999': fora_do_padrao(
                'alimentador', PADRAO_ALIMENTADOR
            ),
            'ocorrências: causa nula ou vazia': F.coalesce(F.trim('causa'), F.lit('')) == '',
            'ocorrências: data de abertura ou encerramento nula': F.col('data_abertura').isNull()
            | F.col('data_encerramento').isNull(),
            'ocorrências: competência diferente do mês da abertura': F.col('competencia').isNull()
            | (F.col('competencia') != F.trunc(F.col('data_abertura').cast('date'), 'month')),
            'ocorrências: duracao_efetiva_min fora de [0, duracao_min]': F.col(
                'duracao_efetiva_min'
            ).isNull()
            | (F.col('duracao_efetiva_min') < 0)
            | (F.col('duracao_efetiva_min') > F.col('duracao_min')),
            'ocorrências: flag de sobreposição ou de evento atípico nula': F.col(
                'flag_sobreposicao'
            ).isNull()
            | F.col('flag_evento_atipico').isNull(),
        },
    )
    violacoes |= contar(
        cadastro,
        {
            'cadastro: id_uc fora do padrão UC + 6 dígitos': fora_do_padrao('id_uc', PADRAO_ID_UC),
            'cadastro: alimentador fora do padrão ALM-AAA-999': fora_do_padrao(
                'alimentador', PADRAO_ALIMENTADOR
            ),
            'cadastro: município nulo ou vazio': F.coalesce(F.trim('municipio'), F.lit('')) == '',
        },
    )

    cadastro_uc = cadastro.select('id_uc', F.col('alimentador').alias('alimentador_cadastro'))
    cruzado = ocorrencias.join(cadastro_uc, 'id_uc', 'left')
    violacoes |= {
        'ocorrências: id_ocorrencia repetido (linhas a mais)': ocorrencias.count()
        - ocorrencias.select('id_ocorrencia').distinct().count(),
        'ocorrências: id_uc ausente do cadastro': cruzado.filter(
            F.col('alimentador_cadastro').isNull()
        ).count(),
        'ocorrências: alimentador diferente do cadastro da UC': cruzado.filter(
            F.col('alimentador') != F.col('alimentador_cadastro')
        ).count(),
        'ocorrências: evento atípico variando dentro de alimentador × competência': (
            ocorrencias.groupBy('alimentador', 'competencia')
            .agg(F.countDistinct('flag_evento_atipico').alias('valores'))
            .filter('valores > 1')
            .count()
        ),
        'cadastro: id_uc repetido (linhas a mais)': cadastro.count()
        - cadastro.select('id_uc').distinct().count(),
        'cadastro: alimentador com mais de um município': (
            cadastro.groupBy('alimentador')
            .agg(F.countDistinct('municipio').alias('municipios'))
            .filter('municipios > 1')
            .count()
        ),
    }
    checar(violacoes, 'fontes')


def construir_gold_continuidade(ocorrencias: DataFrame, cadastro: DataFrame) -> dict:
    """Monta as três tabelas do contrato a partir das Silvers de ocorrências e de cadastro."""
    dim_alimentador = cadastro.groupBy('alimentador').agg(
        F.min('municipio').alias('municipio'),  # um município por alimentador (validado)
        F.count(F.lit(1)).cast(IntegerType()).alias('numero_ucs'),
    )

    ocorrencias_uc = ocorrencias.select(
        'id_ocorrencia',
        'id_uc',
        'alimentador',
        'data_abertura',
        'data_encerramento',
        F.col('duracao_min').cast(DECIMAL_CONTRATO).alias('duracao_min'),
        F.col('duracao_efetiva_min').cast(DECIMAL_CONTRATO).alias('duracao_efetiva_min'),
        'causa',
        'flag_sobreposicao',
    )

    # Calendário completo, da primeira à última competência com ocorrência
    calendario = ocorrencias.agg(
        F.min('competencia').alias('inicio'), F.max('competencia').alias('fim')
    ).select(
        F.explode(F.sequence('inicio', 'fim', F.expr('INTERVAL 1 MONTH'))).alias('competencia')
    )

    # Numeradores: minutos sem energia (sem os minutos em comum) e interrupções distintas
    por_mes = ocorrencias.groupBy('alimentador', 'competencia').agg(
        F.sum('duracao_efetiva_min').alias('minutos_efetivos'),
        F.count(F.when(~F.col('flag_sobreposicao'), 1)).alias('interrupcoes'),
        F.bool_or('flag_evento_atipico').alias('flag_evento_atipico'),
    )

    # Causa principal: mais minutos efetivos; empate vai para mais interrupções e depois o nome
    por_causa = ocorrencias.groupBy('alimentador', 'competencia', 'causa').agg(
        F.sum('duracao_efetiva_min').alias('minutos_causa'),
        F.count(F.when(~F.col('flag_sobreposicao'), 1)).alias('interrupcoes_causa'),
    )
    ordem = Window.partitionBy('alimentador', 'competencia').orderBy(
        F.desc('minutos_causa'), F.desc('interrupcoes_causa'), F.asc('causa')
    )
    causa_principal = (
        por_causa.withColumn('posicao', F.row_number().over(ordem))
        .filter('posicao = 1')
        .select('alimentador', 'competencia', 'causa')
    )

    minutos = F.coalesce(F.col('minutos_efetivos'), F.lit(0)).cast(DecimalType(20, 0))
    interrupcoes = F.coalesce(F.col('interrupcoes'), F.lit(0))
    continuidade = (
        dim_alimentador.select('alimentador', 'numero_ucs')
        .crossJoin(calendario)
        .join(por_mes, ['alimentador', 'competencia'], 'left')
        .join(causa_principal, ['alimentador', 'competencia'], 'left')
        .select(
            'alimentador',
            'competencia',
            'numero_ucs',
            F.round(minutos / 60, 2).cast(DECIMAL_CONTRATO).alias('chi_horas'),
            interrupcoes.cast(IntegerType()).alias('interrupcoes'),
            F.round(minutos / (F.col('numero_ucs') * 60), 2).cast(DECIMAL_CONTRATO).alias('dec'),
            F.round(interrupcoes.cast(DecimalType(20, 0)) / F.col('numero_ucs'), 2)
            .cast(DECIMAL_CONTRATO)
            .alias('fec'),
            F.coalesce(F.col('causa'), F.lit(CAUSA_SEM_OCORRENCIA)).alias('causa'),
            F.coalesce(F.col('flag_evento_atipico'), F.lit(False)).alias('flag_evento_atipico'),
        )
    )
    return {
        'dim_alimentador': dim_alimentador,
        'ocorrencias_uc': ocorrencias_uc,
        'continuidade_alimentador_mensal': continuidade,
    }


def validar_gold(tabelas: dict, ocorrencias: DataFrame, cadastro: DataFrame, rotulo: str) -> None:
    """Contrato, grade e reconciliação das três tabelas contra as Silvers."""
    dim = tabelas['dim_alimentador']
    ocorrencias_uc = tabelas['ocorrencias_uc']
    continuidade = tabelas['continuidade_alimentador_mensal']

    fonte = ocorrencias.agg(
        F.count(F.lit(1)).alias('linhas'),
        F.sum('duracao_min').alias('minutos'),
        F.sum('duracao_efetiva_min').alias('minutos_efetivos'),
        F.count(F.when(~F.col('flag_sobreposicao'), 1)).alias('interrupcoes'),
        F.min('competencia').alias('inicio'),
        F.max('competencia').alias('fim'),
    ).first()
    meses = (
        (fonte['fim'].year - fonte['inicio'].year) * 12
        + fonte['fim'].month
        - fonte['inicio'].month
        + 1
    )
    alimentadores = cadastro.select('alimentador').distinct().count()
    pares_atipicos = (
        ocorrencias.filter('flag_evento_atipico')
        .select('alimentador', 'competencia')
        .distinct()
        .count()
    )
    causas = [r['causa'] for r in ocorrencias.select('causa').distinct().collect()]

    nulo_ou_negativo = F.lit(False)
    for coluna in ('chi_horas', 'interrupcoes', 'dec', 'fec'):
        nulo_ou_negativo = nulo_ou_negativo | F.col(coluna).isNull() | (F.col(coluna) < 0)
    violacoes = contar(
        continuidade,
        {
            'continuidade: alimentador fora do padrão': fora_do_padrao(
                'alimentador', PADRAO_ALIMENTADOR
            ),
            'continuidade: competência nula ou fora do dia 1': F.col('competencia').isNull()
            | (F.dayofmonth('competencia') != 1),
            'continuidade: numero_ucs nulo ou menor que 1': F.col('numero_ucs').isNull()
            | (F.col('numero_ucs') < 1),
            'continuidade: CHI, interrupções, DEC ou FEC nulo ou negativo': nulo_ou_negativo,
            'continuidade: causa fora do domínio': ~F.col('causa').isin(
                *causas, CAUSA_SEM_OCORRENCIA
            )
            | F.col('causa').isNull(),
            'continuidade: flag_evento_atipico nula': F.col('flag_evento_atipico').isNull(),
            'continuidade: DEC diferente de CHI ÷ número de UCs (mais de 0,01)': F.abs(
                F.col('dec') - F.col('chi_horas') / F.col('numero_ucs')
            )
            > 0.01,
            'continuidade: FEC diferente de interrupções ÷ número de UCs (mais de 0,01)': F.abs(
                F.col('fec') - F.col('interrupcoes') / F.col('numero_ucs')
            )
            > 0.01,
            'continuidade: mês sem interrupção com CHI ou causa': (F.col('interrupcoes') == 0)
            & ((F.col('chi_horas') > 0) | (F.col('causa') != CAUSA_SEM_OCORRENCIA)),
        },
    )
    violacoes |= contar(
        ocorrencias_uc,
        {
            'ocorrencias_uc: coluna obrigatória nula': F.greatest(
                *[F.col(c).isNull() for c in ocorrencias_uc.columns]
            ),
            'ocorrencias_uc: id_uc ou alimentador fora do padrão': fora_do_padrao(
                'id_uc', PADRAO_ID_UC
            )
            | fora_do_padrao('alimentador', PADRAO_ALIMENTADOR),
            'ocorrencias_uc: encerramento antes da abertura': F.col('data_encerramento')
            < F.col('data_abertura'),
            'ocorrencias_uc: duracao_efetiva_min fora de [0, duracao_min]': (
                F.col('duracao_efetiva_min') < 0
            )
            | (F.col('duracao_efetiva_min') > F.col('duracao_min')),
        },
    )
    violacoes |= contar(
        dim,
        {
            'dim_alimentador: alimentador fora do padrão': fora_do_padrao(
                'alimentador', PADRAO_ALIMENTADOR
            ),
            'dim_alimentador: município nulo ou vazio': F.coalesce(F.trim('municipio'), F.lit(''))
            == '',
            'dim_alimentador: numero_ucs nulo ou negativo': F.col('numero_ucs').isNull()
            | (F.col('numero_ucs') < 0),
        },
    )

    total = continuidade.agg(
        F.count(F.lit(1)).alias('linhas'),
        F.countDistinct('alimentador', 'competencia').alias('chaves'),
        F.sum('chi_horas').alias('chi_horas'),
        F.sum('interrupcoes').alias('interrupcoes'),
        F.count(F.when(F.col('flag_evento_atipico'), 1)).alias('atipicos'),
    ).first()
    total_ocorrencias = ocorrencias_uc.agg(
        F.count(F.lit(1)).alias('linhas'),
        F.countDistinct('id_ocorrencia').alias('ids'),
        F.sum('duracao_min').alias('minutos'),
        F.sum('duracao_efetiva_min').alias('minutos_efetivos'),
    ).first()
    total_dim = dim.agg(
        F.count(F.lit(1)).alias('linhas'),
        F.countDistinct('alimentador').alias('ids'),
        F.sum('numero_ucs').alias('ucs'),
    ).first()
    # Cada CHI mensal é arredondado em 0,005 h no máximo; a soma pode acumular isso por linha
    diferenca_chi = abs(total['chi_horas'] - Decimal(fonte['minutos_efetivos']) / 60)
    ucs_dim = dim.select('alimentador', F.col('numero_ucs').alias('ucs_dim'))
    violacoes |= {
        'continuidade: alimentador × competência repetido (linhas a mais)': total['linhas']
        - total['chaves'],
        'continuidade: linhas fora da grade alimentadores × competências': abs(
            total['linhas'] - alimentadores * meses
        ),
        'continuidade: interrupções diferentes das ocorrências sem sobreposição': abs(
            total['interrupcoes'] - fonte['interrupcoes']
        ),
        'continuidade: CHI diferente dos minutos efetivos ÷ 60 além do arredondamento': int(
            diferenca_chi > Decimal('0.005') * total['linhas']
        ),
        'continuidade: competências de evento atípico diferentes da Silver': abs(
            total['atipicos'] - pares_atipicos
        ),
        'continuidade: número de UCs diferente da dim_alimentador': continuidade.join(
            ucs_dim, 'alimentador', 'left'
        )
        .filter(~F.col('numero_ucs').eqNullSafe(F.col('ucs_dim')))
        .count(),
        'ocorrencias_uc: linhas diferentes da Silver': abs(
            total_ocorrencias['linhas'] - fonte['linhas']
        ),
        'ocorrencias_uc: id_ocorrencia repetido (linhas a mais)': total_ocorrencias['linhas']
        - total_ocorrencias['ids'],
        'ocorrencias_uc: soma de duracao_min diferente da Silver': int(
            total_ocorrencias['minutos'] != fonte['minutos']
        ),
        'ocorrencias_uc: soma de duracao_efetiva_min diferente da Silver': int(
            total_ocorrencias['minutos_efetivos'] != fonte['minutos_efetivos']
        ),
        'ocorrencias_uc: id_uc ausente do cadastro (dim_uc)': ocorrencias_uc.join(
            cadastro, 'id_uc', 'left_anti'
        ).count(),
        'ocorrencias_uc: alimentador ausente da dim_alimentador': ocorrencias_uc.join(
            dim, 'alimentador', 'left_anti'
        ).count(),
        'dim_alimentador: alimentador repetido (linhas a mais)': total_dim['linhas']
        - total_dim['ids'],
        'dim_alimentador: soma de UCs diferente do cadastro': abs(
            total_dim['ucs'] - cadastro.count()
        ),
    }
    checar(violacoes, rotulo)
    print(
        f'Reconciliação ({rotulo}): {fonte["linhas"]:,} ocorrências | '
        f'{fonte["interrupcoes"]:,} interrupções | {fonte["minutos_efetivos"]:,} minutos '
        f'efetivos = {total["chi_horas"]:,} h de CHI | {alimentadores} alimentadores × '
        f'{meses} competências = {total["linhas"]:,} linhas'
    )


def publicar_delta(spark, tabelas: dict, destinos: dict, validar) -> tuple:
    """Publica e confere as tabelas Gold; recupera os destinos afetados em caso de falha."""
    anteriores = {}
    for nome in tabelas:
        destino = destinos[nome]
        if spark.catalog.tableExists(destino):
            detalhe = spark.sql(f'DESCRIBE DETAIL {destino}').first()
            if detalhe['format'] != 'delta':
                raise RuntimeError(f'O destino precisa ser Delta: {destino}')
            anteriores[nome] = spark.sql(f'DESCRIBE HISTORY {destino} LIMIT 1').first()['version']
        else:
            anteriores[nome] = None

    afetadas = []
    persistidas, versoes = {}, {}
    try:
        for nome, df in tabelas.items():
            destino = destinos[nome]
            afetadas.append(nome)
            (
                df.write.format('delta')
                .mode('overwrite')
                .option('overwriteSchema', 'true')
                .saveAsTable(destino)
            )
            versao = spark.sql(f'DESCRIBE HISTORY {destino} LIMIT 1').first()['version']
            persistido = spark.read.option('versionAsOf', versao).table(destino)
            esperado = [(c.name, c.dataType.simpleString()) for c in df.schema]
            gravado = [(c.name, c.dataType.simpleString()) for c in persistido.schema]
            if esperado != gravado:
                raise RuntimeError(f'Schema persistido divergente: {destino}')
            if (
                df.exceptAll(persistido).limit(1).count()
                or persistido.exceptAll(df).limit(1).count()
            ):
                raise RuntimeError(f'Conteúdo persistido divergente: {destino}')
            persistidas[nome] = persistido
            versoes[nome] = {'tabela': destino, 'versao_delta': versao}
        validar(persistidas)
    except Exception as erro:
        falhas = []
        for nome in reversed(afetadas):
            destino = destinos[nome]
            try:
                versao = anteriores[nome]
                if versao is None:
                    spark.sql(f'DROP TABLE IF EXISTS {destino}')
                else:
                    spark.sql(f'RESTORE TABLE {destino} TO VERSION AS OF {versao}')
            except Exception as falha:
                falhas.append(RuntimeError(f'Falha ao recuperar {destino}: {falha}'))
        if falhas:
            raise ExceptionGroup(
                'Publicação falhou e a recuperação ficou incompleta', [erro, *falhas]
            ) from erro
        raise
    return persistidas, versoes

In [0]:
# ruff: noqa: F821
import re

parametros = {
    'tabela_ocorrencias': 'v-energy.silver.tb_ocorrencias_rede',
    'tabela_cadastro': 'v-energy.silver.tb_unidades_consumidoras',
    'schema_gold': 'v-energy.gold',
    'modo': 'validar',
}
for nome, padrao in parametros.items():
    dbutils.widgets.text(nome, padrao)
config = {nome: dbutils.widgets.get(nome).strip() for nome in parametros}
if config['modo'] not in {'validar', 'publicar'}:
    raise RuntimeError('modo deve ser validar ou publicar')


def qualificar(nome, partes=3):
    segmentos = nome.split('.')
    if len(segmentos) != partes or any(
        not re.fullmatch(r'[A-Za-z0-9_-]+', segmento) for segmento in segmentos
    ):
        raise RuntimeError(f'Identificador inválido: {nome}')
    return '.'.join(f'`{segmento}`' for segmento in segmentos)


schema_gold = qualificar(config['schema_gold'], 2)
if config['schema_gold'].split('.')[1] in {'silver', 'bronze', 'default'}:
    raise RuntimeError('O destino deve ser um schema próprio da Gold')
fontes = {}
print(f'Schema Gold: {schema_gold} | Modo: {config["modo"]}')
if config['modo'] == 'publicar':
    print(f'ATENÇÃO: modo=publicar. As tabelas Gold serão gravadas em {schema_gold}')

Schema Gold: `v-energy`.`gold` | Modo: validar


## 1. Leitura consistente da Silver (`VERSION AS OF`) e validação

Fixa a versão Delta atual das duas Silvers no início da execução, para que validações, transformações e escrita leiam exatamente os mesmos dados. Antes de montar a Gold, confere colunas, chaves, domínios e o cruzamento entre ocorrências e cadastro: toda UC interrompida existe no cadastro e está no mesmo alimentador. Depois de montar, confere o contrato das três tabelas, a grade alimentadores × competências e a reconciliação com a Silver.

In [0]:
# ruff: noqa: F821
def ler_snapshot(chave):
    tabela = qualificar(config[chave])
    if not spark.catalog.tableExists(tabela):
        raise RuntimeError(
            f'Tabela Silver não encontrada: {tabela}. '
            f'Publique a Silver correspondente ou ajuste o widget {chave}.'
        )
    detalhe = spark.sql(f'DESCRIBE DETAIL {tabela}').first()
    if detalhe['format'] != 'delta':
        raise RuntimeError(f'A fonte precisa ser Delta: {tabela}')
    historico = spark.sql(f'DESCRIBE HISTORY {tabela} LIMIT 1').first()
    fontes[chave] = {
        'tabela': config[chave],
        'id_delta': detalhe['id'],
        'versao_delta': historico['version'],
    }
    print(f'{chave}: {tabela} (versão {historico["version"]})')
    return spark.read.option('versionAsOf', historico['version']).table(tabela)


df_ocorrencias = ler_snapshot('tabela_ocorrencias')
df_cadastro = ler_snapshot('tabela_cadastro')

validar_fontes(df_ocorrencias, df_cadastro)
tabelas = construir_gold_continuidade(df_ocorrencias, df_cadastro)
validar_gold(tabelas, df_ocorrencias, df_cadastro, 'antes de gravar')
for nome, df_tab in tabelas.items():
    print(f'[OK] {nome}: {df_tab.count():,} registros validados em memória')

# Informativo: a data de ligação não entra na regra (docs/gold/continuidade.md, seção 3)
ligacao = (
    df_ocorrencias.join(df_cadastro.select('id_uc', 'data_ligacao'), 'id_uc')
    .agg(
        F.count(
            F.when(F.col('data_abertura') < F.col('data_ligacao').cast('timestamp_ntz'), 1)
        ).alias('antes'),
        F.count(F.when(F.col('data_ligacao').isNull(), 1)).alias('sem_data'),
    )
    .first()
)
print(
    f'Informativo: {ligacao["antes"]:,} ocorrências abertas antes da data_ligacao da UC e '
    f'{ligacao["sem_data"]:,} de UCs sem data_ligacao; todas entram no DEC e no FEC.'
)

tabela_ocorrencias: `v-energy`.`silver`.`tb_ocorrencias_rede` (versão 1)
tabela_cadastro: `v-energy`.`silver`.`tb_unidades_consumidoras` (versão 41)
[OK   ] (fontes) ocorrências: id_ocorrencia nulo: 0
[OK   ] (fontes) ocorrências: id_uc fora do padrão UC + 6 dígitos: 0
[OK   ] (fontes) ocorrências: alimentador fora do padrão ALM-AAA-999: 0
[OK   ] (fontes) ocorrências: causa nula ou vazia: 0
[OK   ] (fontes) ocorrências: data de abertura ou encerramento nula: 0
[OK   ] (fontes) ocorrências: competência diferente do mês da abertura: 0
[OK   ] (fontes) ocorrências: duracao_efetiva_min fora de [0, duracao_min]: 0
[OK   ] (fontes) ocorrências: flag de sobreposição ou de evento atípico nula: 0
[OK   ] (fontes) cadastro: id_uc fora do padrão UC + 6 dígitos: 0
[OK   ] (fontes) cadastro: alimentador fora do padrão ALM-AAA-999: 0
[OK   ] (fontes) cadastro: município nulo ou vazio: 0
[OK   ] (fontes) ocorrências: id_ocorrencia repetido (linhas a mais): 0
[OK   ] (fontes) ocorrências: id_uc ausen

## 2. Verificação da pergunta da Gerente de Operação

**Beatriz Farias:** _"Quero acompanhar esses indicadores por alimentador ao longo do tempo e identificar os piores, para direcionar a manutenção preventiva."_ Confere nas tabelas geradas:
- DEC e FEC de todos os alimentadores em cada ciclo de 12 meses (julho a junho), com e sem o evento atípico;
- os piores alimentadores no período inteiro por CHI, o critério do glossário para priorizar manutenção, com e sem o evento atípico;
- o CHI por causa, a partir de `ocorrencias_uc`.

In [0]:
# ruff: noqa: F821
continuidade = tabelas['continuidade_alimentador_mensal']
total_ucs = tabelas['dim_alimentador'].agg(F.sum('numero_ucs')).first()[0]
sem_evento = ~F.col('flag_evento_atipico')


def por_uc(soma):
    return F.round(soma / total_ucs, 2).cast(DECIMAL_CONTRATO)


print(f'=== 1) DEC e FEC de todos os alimentadores por ciclo julho-junho ({total_ucs:,} UCs) ===')
inicio_ciclo = F.year(F.add_months('competencia', -6))
display(
    continuidade.groupBy(
        F.concat(
            inicio_ciclo.cast('string'),
            F.lit('-07 a '),
            (inicio_ciclo + 1).cast('string'),
            F.lit('-06'),
        ).alias('ciclo')
    )
    .agg(
        por_uc(F.sum('chi_horas')).alias('dec'),
        por_uc(F.sum('interrupcoes')).alias('fec'),
        por_uc(F.sum(F.when(sem_evento, F.col('chi_horas')))).alias('dec_sem_evento_atipico'),
        por_uc(F.sum(F.when(sem_evento, F.col('interrupcoes')))).alias('fec_sem_evento_atipico'),
    )
    .orderBy('ciclo')
)


def piores_alimentadores(df, rotulo):
    print(f'=== 2) Piores alimentadores no período inteiro, por CHI ({rotulo}) ===')
    display(
        df.groupBy('alimentador')
        .agg(
            F.max('numero_ucs').alias('numero_ucs'),
            F.sum('chi_horas').alias('chi_horas'),
            F.sum('interrupcoes').alias('interrupcoes'),
        )
        .withColumn(
            'dec', F.round(F.col('chi_horas') / F.col('numero_ucs'), 2).cast(DECIMAL_CONTRATO)
        )
        .withColumn(
            'fec', F.round(F.col('interrupcoes') / F.col('numero_ucs'), 2).cast(DECIMAL_CONTRATO)
        )
        .orderBy(F.desc('chi_horas'), 'alimentador')
        .limit(10)
    )


piores_alimentadores(continuidade, 'com o evento atípico')
piores_alimentadores(continuidade.filter(sem_evento), 'sem o evento atípico')

print('=== 3) CHI e interrupções por causa (ocorrencias_uc) ===')
display(
    tabelas['ocorrencias_uc']
    .groupBy('causa')
    .agg(
        F.round(F.sum('duracao_efetiva_min') / 60, 2).alias('chi_horas'),
        F.count(F.when(~F.col('flag_sobreposicao'), 1)).alias('interrupcoes'),
    )
    .orderBy(F.desc('chi_horas'))
)

=== 1) DEC e FEC de todos os alimentadores por ciclo julho-junho (45,000 UCs) ===


ciclo,dec,fec,dec_sem_evento_atipico,fec_sem_evento_atipico
2022-07 a 2023-06,1.47,0.89,1.47,0.89
2023-07 a 2024-06,1.50,0.90,1.50,0.90
2024-07 a 2025-06,1.50,0.90,1.50,0.90
2025-07 a 2026-06,3.43,1.30,2.05,1.02


=== 2) Piores alimentadores no período inteiro, por CHI (com o evento atípico) ===


alimentador,numero_ucs,chi_horas,interrupcoes,dec,fec
ALM-REC-008,2285,35645.89,8063,15.60,3.53
ALM-SAO-004,1091,18361.68,4602,16.83,4.22
ALM-JAB-004,863,14826.18,3868,17.18,4.48
ALM-REC-014,339,2644.53,1456,7.80,4.29
ALM-CAM-003,205,2628.89,1473,12.82,7.19
ALM-JAB-013,546,2626.33,1392,4.81,2.55
ALM-JAB-017,646,2587.34,1409,4.01,2.18
ALM-OLI-009,286,2578.62,1403,9.02,4.91
ALM-CAB-002,358,2577.96,1412,7.20,3.94
ALM-CAM-004,147,2560.97,1423,17.42,9.68


=== 2) Piores alimentadores no período inteiro, por CHI (sem o evento atípico) ===


alimentador,numero_ucs,chi_horas,interrupcoes,dec,fec
ALM-REC-014,339,2644.53,1456,7.80,4.29
ALM-CAM-003,205,2628.89,1473,12.82,7.19
ALM-JAB-013,546,2626.33,1392,4.81,2.55
ALM-JAB-017,646,2587.34,1409,4.01,2.18
ALM-OLI-009,286,2578.62,1403,9.02,4.91
ALM-CAB-002,358,2577.96,1412,7.20,3.94
ALM-CAM-004,147,2560.97,1423,17.42,9.68
ALM-REC-003,270,2559.46,1401,9.48,5.19
ALM-REC-002,435,2557.46,1423,5.88,3.27
ALM-REC-033,334,2548.28,1356,7.63,4.06


=== 3) CHI e interrupções por causa (ocorrencias_uc) ===


causa,chi_horas,interrupcoes
CLIMATICA,128249.15,43421
SOBRECARGA,44124.52,26318
VEGETACAO,43792.08,26143
NAO IDENTIFICADA,43401.77,26319
TERCEIROS,42975.55,26137
FALHA DE EQUIPAMENTO,42792.42,25938
NAO INFORMADA,10473.85,5394


## 3. Publicação Delta com `RESTORE`

A publicação só ocorre quando `modo=publicar`. Antes da primeira escrita, registra a versão atual de cada tabela de destino. Depois de gravar, confere o schema e o conteúdo gravados e repete a validação da seção 1 sobre as tabelas persistidas; se a escrita ou a conferência falhar, restaura os destinos afetados com `RESTORE TABLE ... TO VERSION AS OF` (ou `DROP TABLE IF EXISTS` na primeira carga).

In [0]:
# ruff: noqa: F821
if config['modo'] == 'validar':
    print('Validação concluída. Nenhum schema ou tabela foi gravado.')
else:
    if any(dbutils.widgets.get(nome).strip() != valor for nome, valor in config.items()):
        raise RuntimeError(
            'Parâmetros mudaram durante a execução. Execute novamente desde o início.'
        )
    destinos = {nome: f'{schema_gold}.`{nome}`' for nome in tabelas}
    for chave, fonte in fontes.items():
        atual = spark.sql(f'DESCRIBE DETAIL {qualificar(config[chave])}').first()
        if atual['id'] != fonte['id_delta']:
            raise RuntimeError(f'A identidade da tabela fonte mudou: {chave}')
    if set(destinos.values()) & {qualificar(config[c]) for c in fontes}:
        raise RuntimeError('Uma tabela de destino coincide com uma fonte')

    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {schema_gold}')
    persistidas, versoes_gold = publicar_delta(
        spark,
        tabelas,
        destinos,
        lambda gravadas: validar_gold(gravadas, df_ocorrencias, df_cadastro, 'tabela gravada'),
    )
    for nome, info in versoes_gold.items():
        print(f'Publicada: {info["tabela"]} (versão {info["versao_delta"]})')

Validação concluída. Nenhum schema ou tabela foi gravado.
